# Mitigating Hallucinations in Large Audio-Language Models — Hands-on Tutorial

Welcome! Large Audio-Language Models (LALMs) can *transcribe*, *describe* and *reason* about sound — but they also **hallucinate**: they confidently answer about sounds that are **not** in the audio. A classic trigger: the speaker *says* "listen, it's thundering!", so the model reports that it **heard thunder**, even though there is no thunder sound at all.

In this tutorial you will:

1. Run an open LALM — **Qwen2.5-Omni** — on a handful of audio clips.
2. Watch it hallucinate in the **vanilla** setting (no mitigation).
3. Apply a training-free, decoding-time mitigation called **AAD (Audio-Aware Decoding)** — and write its core contrastive formula yourself.
4. Score the free-text answers automatically with an **LLM-as-a-judge** (Qwen3.5) and compare against the ground truth.

Everything is **self-contained in this notebook** (no project code is imported). A single GPU with ~18 GB is enough.

> **The data.** `.cache/aha_workshop/samples.jsonl` holds 6 questions over 5 short clips, in the same schema as the full AHa-Bench. The clips are picked to trigger hallucinations, so the ground-truth answer is usually **"no"**.

In [ ]:
# --- Imports & global config ------------------------------------------------
import json
import librosa
import torch
import torch.nn.functional as F
import numpy as np
from transformers import (
    set_seed,
    LogitsProcessor,
    AutoProcessor,
    AutoModelForMultimodalLM,
    Qwen2_5OmniProcessor,
    Qwen2_5OmniThinkerForConditionalGeneration,
)
from pathlib import Path
from IPython.display import Audio, display
from huggingface_hub import snapshot_download

# A fixed seed so everyone in the room sees the same outputs.
set_seed(42)

DEVICE = "cuda"
SAMPLING_RATE = 16000

print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# Download the model weights from HuggingFace Hub (if not already cached)
# Qwen2_5OmniThinkerForConditionalGeneration.from_pretrained("Qwen/Qwen2.5-Omni-3B")
# AutoModelForMultimodalLM.from_pretrained("Qwen/Qwen3.5-9B")

In [ ]:
# --- Download the dataset ----------------------------------------------------
REPO_ID = "soizhiwen/aha-workshop"
CACHE_DIR = Path(".cache") / "aha_workshop"

local_path = snapshot_download(
    repo_id=REPO_ID, repo_type="dataset", local_dir=CACHE_DIR
)
print(f"Dataset downloaded to: {local_path}")

In [ ]:
# --- Load the mini benchmark ------------------------------------------------
# Each line of samples.jsonl is one question. The fields we use:
#   audio_path : path to a 16 kHz wav clip
#   question   : the yes/no question asked about the audio
#   answer     : the ground-truth answer
#   type       : which hallucination category the sample probes
#   audio_text : the spoken words in the clip (if any), for your reference
with open(CACHE_DIR / "samples.jsonl") as f:
    SAMPLES = [json.loads(line) for line in f]

print(f"Loaded {len(SAMPLES)} samples\n")
for s in SAMPLES:
    print(f"[{s['type']:<16}] GT={s['answer']:<3} | {s['question']}")


def to_msg(sample: dict) -> dict:
    """Turn a raw dataset row into the `msg` dict the model wrappers expect.

    The model only needs the audio path(s) and the question text.
    `audio` is always a list because some tasks compare several clips.
    """
    return {
        "index": sample["index"],
        "audio": [sample["audio_path"]],
        "text": sample["question"],
        "answer": sample["answer"],
        "meta": {"type": sample["type"]},
    }


def listen(sample: dict):
    """Play a clip inline so you can hear exactly what the model hears."""
    print(f"Q: {sample['question']}   (ground truth: {sample['answer']})")
    if sample.get("audio_text", "N/A") not in ("N/A", ""):
        print(f'Spoken words: "{sample["audio_text"]}"')
    display(Audio(sample["audio_path"]))

In [ ]:
# Listen first! Notice that the speaker *mentions* thunder, but you never
# actually hear a thunderclap — the sound itself is just a voice.
listen(SAMPLES[3])

## The decoding-time toolkit: contrastive decoding

**AAD (Audio-Aware Decoding)** is built on one idea — **contrastive decoding**. At each generation step we run the model **twice**:

- a **positive** branch that sees the real audio → logits `logit_pos` (these arrive as `scores`)
- a **negative** branch that sees the **same prompt with the audio removed** → logits `logit_neg`

We then steer the output *away* from whatever the audio-blind negative branch wants:

```
final_logits = (1 + α) · logit_pos − α · logit_neg
```

Tokens the model would pick **even without listening** (hallucinations, language priors) get suppressed; tokens that genuinely depend on the audio get amplified. `α` (`alpha`) controls the strength.

The class below is a HuggingFace `LogitsProcessor`: `generate()` calls it once per step with the positive logits (`scores`), and we return the modified logits. We drive the negative branch's own forward pass + KV cache by hand. **Your job:** write the one-line contrastive formula (marked in the cell).

In [ ]:
class AADLogitsProcessor(LogitsProcessor):
    """Audio-Aware Decoding: negative branch = the prompt with NO audio.
    Shared logic: final = (1 + alpha) * logit_pos - alpha * logit_neg.

    HuggingFace's `generate()` drives the *positive* branch for us and hands us
    its logits as `scores`. We are responsible for the *negative* branch: we keep
    our own forward pass and KV cache, fed the negative inputs.
    """

    def __init__(self, model, inputs_neg, alpha=1.0):
        self.model = model
        self.alpha = alpha
        self.inputs_neg = inputs_neg  # the negative context (dict of tensors)
        self.atts_neg = inputs_neg["attention_mask"]
        self.past_key_values_neg = None  # KV cache for the negative branch
        self.first_call = True

    def __call__(self, input_ids, scores):
        with torch.no_grad():
            if self.first_call:
                # Step 0: run the full negative prompt once and cache its KV states.
                self.first_call = False
                out_neg = self.model(
                    **self.inputs_neg, use_cache=True, return_dict=True
                )
                self.past_key_values_neg = out_neg.past_key_values
            else:
                # Later steps: the two branches share the same generated text, so we
                # only need to feed the negative branch the single most-recent token
                # (reusing its cache).
                last_token = input_ids[:, -1:]
                self.atts_neg = torch.cat(
                    [
                        self.atts_neg,
                        torch.ones_like(last_token, dtype=self.atts_neg.dtype),
                    ],
                    dim=1,
                )
                # Tell the model where in the cache this token sits (needed when
                # we drive the forward pass manually like this).
                past = self.past_key_values_neg.get_seq_length()
                cache_position = torch.arange(past, past + 1, device=last_token.device)
                out_neg = self.model(
                    input_ids=last_token,
                    attention_mask=self.atts_neg,
                    past_key_values=self.past_key_values_neg,
                    cache_position=cache_position,
                    use_cache=True,
                    return_dict=True,
                )
                self.past_key_values_neg = out_neg.past_key_values

            logits_neg = out_neg.logits[:, -1, :]  # (batch, vocab)

        # ============== >>> WRITE THE CONTRASTIVE FORMULA HERE <<< =============
        modified_logits = None  # TODO: replace None
        # ======================================================================
        if modified_logits is None:
            raise NotImplementedError(
                "Write the contrastive formula above using scores, logits_neg, self.alpha."
            )
        return modified_logits


print("Contrastive logits processors ready.")

## The model wrapper

The wrapper exposes a **single** entry point, `generate(msg, method=...)`, with two modes:

| `method`   | what happens                                          |
|------------|-------------------------------------------------------|
| `"vanilla"`| plain generation, no mitigation                       |
| `"aad"`    | Audio-Aware Decoding (negative branch = no audio)     |

The AAD path just attaches the `AADLogitsProcessor` from above to `generate()`.

In [ ]:
class Qwen2_5Omni:
    """Self-contained wrapper around Qwen2.5-Omni."""

    NAME = "Qwen2.5-Omni-3B"
    MAX_NEW_TOKENS = 512
    SYSTEM_PROMPT = (
        "You are Qwen, a virtual human developed by the Qwen Team, Alibaba Group, "
        "capable of perceiving auditory and visual inputs, as well as generating "
        "text and speech."
    )

    def __init__(self, model_path="Qwen/Qwen2.5-Omni-3B"):
        self.processor = Qwen2_5OmniProcessor.from_pretrained(model_path)
        self.model = Qwen2_5OmniThinkerForConditionalGeneration.from_pretrained(
            model_path, dtype="auto", device_map="auto"
        ).eval()
        torch.cuda.empty_cache()

    def load_audio(self, path) -> np.ndarray:
        wav, _ = librosa.load(path, sr=SAMPLING_RATE, mono=True)
        return wav

    def build_inputs(self, msg, audio, prefix=""):
        """`audio` is a waveform (np.ndarray) to condition on, or None for the
        no-audio (negative) branch used by AAD."""
        text = f"{prefix} {msg['text']}".strip() if prefix else msg["text"]
        user_content = [{"type": "text", "text": text}]
        if audio is not None:
            user_content.append({"type": "audio", "audio": "placeholder"})
        messages = [
            {
                "role": "system",
                "content": [{"type": "text", "text": self.SYSTEM_PROMPT}],
            },
            {"role": "user", "content": user_content},
        ]
        chat = self.processor.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=False
        )
        if audio is not None:
            inputs = self.processor(
                text=[chat],
                audio=audio,
                return_tensors="pt",
                padding=True,
                sampling_rate=SAMPLING_RATE,
            )
        else:
            inputs = self.processor(
                text=[chat], audio=None, return_tensors="pt", padding=True
            )
        # .to(dtype) only casts float tensors, so input_ids stay as long ints.
        return inputs.to(DEVICE).to(self.model.dtype)

    def gen_kwargs(self):
        tok = self.processor.tokenizer
        return dict(
            do_sample=False,
            max_new_tokens=self.MAX_NEW_TOKENS,
            pad_token_id=tok.pad_token_id,
            bos_token_id=tok.bos_token_id,
            eos_token_id=tok.eos_token_id,
        )

    def decode(self, output_ids, inputs):
        gen = output_ids[:, inputs["input_ids"].shape[1] :]  # drop the prompt tokens
        return self.processor.batch_decode(
            gen, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )[0]

    @torch.inference_mode()
    def generate(self, msg, method="vanilla", alpha=1.0):
        path = msg["audio"][0]
        wav = self.load_audio(path)
        processor = None

        if method == "aad":
            # A prefix nudges the model to rely on the audio; the negative branch
            # is the SAME prompt with the audio removed.
            prefix = "Focus on the given audio and answer the following question:"
            inputs = self.build_inputs(msg, audio=wav, prefix=prefix)
            inputs_neg = self.build_inputs(msg, audio=None, prefix=prefix)
            processor = AADLogitsProcessor(self.model, inputs_neg, alpha=alpha)

        else:  # vanilla
            inputs = self.build_inputs(msg, audio=wav)

        out = self.model.generate(
            **inputs,
            **self.gen_kwargs(),
            logits_processor=[processor] if processor else None,
        )
        return self.decode(out, inputs)


print("Qwen2_5Omni wrapper defined.")

In [ ]:
def compare(model, samples, methods=("vanilla", "aad")):
    """Run `model` on every sample under each decoding method and print answers."""
    results = []
    for s in samples:
        msg = to_msg(s)
        display(Audio(s["audio_path"]))  # play the clip the model is judging
        print(f"[{s['type']}]  Q: {s['question']}")
        print(f"   ground truth : {s['answer']}")
        preds = {}
        for m in methods:
            preds[m] = model.generate(msg, method=m)
            print(f"   {m:<8}     : {preds[m]}")
        print()
        results.append((s, preds))
    return results


print("compare() helper ready.")

## Part 1 — Qwen2.5-Omni

Loading the model the first time downloads the weights from Hugging Face (a few minutes / several GB). After that it is cached locally.

In [ ]:
omni = Qwen2_5Omni()
print("Loaded:", omni.NAME)

### 1a. Vanilla — no mitigation

Run the base model with no tricks. **Watch for hallucinations**: the ground truth is "no" for most clips, but the model often answers "yes" — e.g. it claims to hear thunder or a dog simply because those words were *spoken*.

In [ ]:
_ = compare(omni, SAMPLES, methods=("vanilla",))

### 1b. Add AAD (Audio-Aware Decoding)

Now turn on AAD. The negative branch reruns each step with the **audio removed**, and we subtract its logits. Answers that came from the *spoken words / language prior* rather than the *actual sound* should get suppressed. Compare the two columns.

We keep the predictions in `results` so the LLM judge can score them next.

In [ ]:
# `results` = [(sample, {"vanilla": pred, "aad": pred}), ...]
results = compare(omni, SAMPLES, methods=("vanilla", "aad"))
del omni
torch.cuda.empty_cache()

## Part 2 — Scoring with an LLM judge

The model answers in free text ("Yes, there is thunder.", "No, only speech is audible.", ...), so to *measure* accuracy we must map each answer to **Yes / No / Unknown** and compare with the ground truth.

For the tutorial we use **`Qwen/Qwen3.5-9B`** instead of the large judge in the project. We hand it the same rubric: given the question and the model's answer, reply with a single word.

In [ ]:
JUDGE_MODEL = "Qwen/Qwen3.5-9B"

JUDGE_PROMPT = (
    "You are an AI assistant responsible for assessing the alignment of an answer with three predefined response options: Yes, No, Unknown.\n"
    "Your objective is to evaluate the given question-answer pair and determine which of the three options (Yes, No, Unknown) best represents the answer.\n"
    "- If the answer clearly aligns with an affirmative response, output 'Yes'.\n"
    "- If the answer clearly aligns with a negative response, output 'No'.\n"
    "- If the answer is ambiguous or does not sufficiently align with either Yes or No, output 'Unknown'.\n"
    "Your output must consist of a single word: 'Yes', 'No', or 'Unknown'.\n"
    "Examples:\n"
    "1. Question: Is the car moving fast?\n"
    "Answer: Yes, it is speeding down the highway.\n"
    "Output: Yes\n"
    "2. Question: Is the dog barking?\n"
    "Answer: The dog is lying quietly on the floor.\n"
    "Output: No\n"
    "3. Question: Is it raining outside?\n"
    "Answer: I don't hear any rain, but it could have rained earlier.\n"
    "Output: Unknown\n"
    "Input:\n"
    "Question: {question}\n"
    "Answer: {answer}\n"
    "Output:"
)

VALID_LABELS = {"yes", "no", "unknown"}


def normalize_label(text: str) -> str:
    """Map the judge's raw output to yes / no / unknown."""
    label = text.strip().lower()
    if label not in VALID_LABELS:
        raise ValueError(f"Unexpected judge output: {text!r} (normalized to {label!r})")
    return label


class QwenJudge:
    """Local Qwen3 used as a Yes/No/Unknown judge (batched)."""

    def __init__(self, max_new_tokens=8):
        self.max_new_tokens = max_new_tokens
        self.processor = AutoProcessor.from_pretrained(JUDGE_MODEL)
        # Left-pad so a batch of different-length prompts all end at the same spot.
        self.processor.tokenizer.padding_side = "left"
        self.model = AutoModelForMultimodalLM.from_pretrained(
            JUDGE_MODEL,
            dtype="auto",
            device_map="auto",
        ).eval()
        torch.cuda.empty_cache()

    def _build_prompt(self, question, answer):
        prompt = JUDGE_PROMPT.format(question=question, answer=answer)
        messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
        # Qwen3 emits <think> blocks by default; disable so the output is a single word.
        try:
            return self.processor.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False,
            )
        except TypeError:
            return self.processor.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )

    @torch.inference_mode()
    def judge_batch(self, qa_pairs):
        """qa_pairs: list of (question, answer) -> list of 'yes'/'no'/'unknown'."""
        prompts = [self._build_prompt(q, a) for q, a in qa_pairs]
        inputs = self.processor(text=prompts, return_tensors="pt", padding=True).to(
            self.model.device
        )
        out = self.model.generate(
            **inputs,
            max_new_tokens=self.max_new_tokens,
            do_sample=False,
            pad_token_id=self.processor.tokenizer.pad_token_id
            or self.processor.tokenizer.eos_token_id,
        )
        new_tokens = out[:, inputs["input_ids"].size(1) :]  # drop the prompt
        decoded = self.processor.batch_decode(new_tokens, skip_special_tokens=True)
        return [normalize_label(t) for t in decoded]


judge = QwenJudge()
print("Loaded judge:", JUDGE_MODEL)

### Judged output vs. ground truth

For every prediction we ask the judge for a Yes/No/Unknown label, then check it against the ground truth. The accuracy line at the bottom tells you whether AAD actually helped.

In [ ]:
methods = ("vanilla", "aad")

# Judge all predictions for each method in one batch.
judged = {
    m: judge.judge_batch([(s["question"], preds[m]) for s, preds in results])
    for m in methods
}

correct = {m: 0 for m in methods}
for i, (s, preds) in enumerate(results):
    gt = s["answer"].strip().lower()
    display(Audio(s["audio_path"]))  # play the clip so you can verify the judge
    print(f"[{s['type']}] Q: {s['question']}")
    print(f"   ground truth : {gt}")
    for m in methods:
        label = judged[m][i]
        ok = "correct" if label == gt else "WRONG"
        correct[m] += label == gt
        print(f'   {m:<8}     : judged={label:<8} ({ok})   <- "{preds[m]}"')
    print()

n = len(results)
print("Accuracy vs Ground Truth:")
for m in methods:
    print(f"   {m:<8}: {correct[m]}/{n} = {correct[m] / n:.0%}")

del judge
torch.cuda.empty_cache()

## Wrap-up

You've seen a complete, training-free hallucination-mitigation pipeline:

- **Vanilla** decoding hallucinates sounds implied by speech or priors.
- **AAD** contrasts against a *no-audio* branch to suppress ungrounded answers — and you wrote the core contrastive formula yourself.
- An **LLM judge** (Qwen3.5-9B) turns free-text answers into Yes/No/Unknown so accuracy can be measured against the ground truth.